In [12]:
import pandas as pd
import json
import os
import glob
from collections import Counter

data_folder = r"C:\Research\EndpointXAI\data"
# Only scan the original Security-Datasets folder, not the duplicate
data_folder = r"C:\Research\EndpointXAI\data\Security-Datasets"
all_files = glob.glob(data_folder + "/**/*.json", recursive=True)
print(f"Found {len(all_files)} files in Security-Datasets only")
print(f"Found {len(all_files)} files")

Found 13 files in Security-Datasets only
Found 13 files


In [13]:
# Scan all files to find every possible field
all_keys = Counter()

for file in all_files:
    try:
        with open(file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    record = json.loads(line)
                    for key in record.keys():
                        all_keys[key] += 1
    except Exception as e:
        pass

# Show top 30 most common fields
print("Top 30 fields across ALL files:")
for key, count in all_keys.most_common(30):
    print(f"  {key}: {count} records")

Top 30 fields across ALL files:
  @timestamp: 45622 records
  host: 45622 records
  port: 28232 records
  Severity: 28232 records
  EventTime: 28232 records
  SourceName: 28232 records
  Channel: 28232 records
  EventReceivedTime: 28232 records
  EventID: 28232 records
  SeverityValue: 28232 records
  Keywords: 28232 records
  EventType: 28232 records
  SourceModuleName: 28232 records
  SourceModuleType: 28232 records
  RecordNumber: 28232 records
  ThreadID: 28232 records
  Task: 28232 records
  Hostname: 28232 records
  @version: 28232 records
  OpcodeValue: 27890 records
  ProviderGuid: 27890 records
  Version: 27890 records
  Message: 27823 records
  Opcode: 26601 records
  Category: 26544 records
  ExecutionProcessID: 25993 records
  tags: 25993 records
  ProcessId: 24735 records
  AccountName: 24441 records
  AccountType: 24441 records


In [14]:
# Save full field list to a text file so we can see everything
with open(r"C:\Research\EndpointXAI\data\all_fields.txt", "w") as f:
    f.write("All fields across all files:\n\n")
    for key, count in all_keys.most_common():
        f.write(f"{key}: {count} records\n")

print(f"Total unique fields: {len(all_keys)}")
print(f"\nFull list saved to all_fields.txt")
print(f"\nFields with 50,000+ records:")
for key, count in all_keys.most_common():
    if count >= 50000:
        print(f"  {key}: {count}")

Total unique fields: 238

Full list saved to all_fields.txt

Fields with 50,000+ records:


In [15]:
# Rebuild dataset - ONLY from real attack data folders
all_records = []

# Filter to only include files in datasets/atomic/windows/
attack_files = [f for f in all_files if 'datasets' in f.replace('\\','/') 
                and 'atomic' in f.replace('\\','/') 
                and 'windows' in f.replace('\\','/')]

print(f"Filtered to {len(attack_files)} actual attack data files")

# CLEAN feature set - leakage columns removed
# Removed: Hostname (100%), Message (100%), ProcessId (99.3%), 
#          ThreadID (94.9%), Opcode (75%), OpcodeValue (75%), EventType (75%)

useful_fields = [
    'EventID',           # 67.7% - meaningful event type
    'Task',              # 56.2% - what action occurred
    'Channel',           # 62.5% - log source
    'SourceName',        # 62.5% - sysmon vs security log
    'SourceModuleName',  # 50% - log provider
    'Severity',          # 60% - event severity
    'SeverityValue',     # 60% - numeric severity
    'AccountName',       # 50% - user account
    'AccountType',       # 33.3% - user/system/admin
    'Keywords'           # 50% - event keywords
]

skipped_count = 0
for file in attack_files:
    try:
        parts = file.replace('\\', '/').split('/')
        try:
            windows_idx = parts.index('windows')
            folder_name = parts[windows_idx + 1]
        except:
            folder_name = 'unknown'
        
        with open(file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        record = json.loads(line)
                        clean = {field: record.get(field, None) for field in useful_fields}
                        clean['attack_label'] = folder_name
                        all_records.append(clean)
                    except:
                        continue
    except Exception as e:
        skipped_count += 1

df = pd.DataFrame(all_records)
print(f"\nTotal events: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"Skipped files: {skipped_count}")
print(f"\nAttack label distribution:")
print(df['attack_label'].value_counts())

Filtered to 6 actual attack data files

Total events: 45622
Total columns: 11
Skipped files: 0

Attack label distribution:
attack_label
lateral_movement    25993
defense_evasion     17390
persistence          2239
Name: count, dtype: int64


In [16]:
from sklearn.preprocessing import LabelEncoder

# Fill missing values
df = df.fillna('unknown')

# Convert all text columns to numbers
encoders = {}
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str)
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        encoders[col] = le

print("All columns converted to numbers ✅")
print(f"\nFinal shape: {df.shape}")
print(df.head())

All columns converted to numbers ✅

Final shape: (45622, 11)
   EventID  Task  Channel SourceName SourceModuleName Severity  SeverityValue  \
0       64    47  unknown    unknown          unknown  unknown              4   
1       64    47  unknown    unknown          unknown  unknown              4   
2       64    47  unknown    unknown          unknown  unknown              4   
3       64    47  unknown    unknown          unknown  unknown              4   
4       64    47  unknown    unknown          unknown  unknown              4   

  AccountName AccountType  Keywords     attack_label  
0     unknown     unknown         7  defense_evasion  
1     unknown     unknown         7  defense_evasion  
2     unknown     unknown         7  defense_evasion  
3     unknown     unknown         7  defense_evasion  
4     unknown     unknown         7  defense_evasion  


In [17]:
from sklearn.model_selection import train_test_split

# Save the new rich dataset
df.to_csv(r"C:\Research\EndpointXAI\data\mordor_rich_features.csv", index=False)
print("Saved rich dataset ✅")

# Split for training
X = df.drop('attack_label', axis=1)
y = df['attack_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Overwrite the old splits with rich features
X_train.to_csv(r"C:\Research\EndpointXAI\data\X_train.csv", index=False)
X_test.to_csv(r"C:\Research\EndpointXAI\data\X_test.csv", index=False)
y_train.to_csv(r"C:\Research\EndpointXAI\data\y_train.csv", index=False)
y_test.to_csv(r"C:\Research\EndpointXAI\data\y_test.csv", index=False)

print(f"\nNew train shape: {X_train.shape}")
print(f"New test shape: {X_test.shape}")
print("\n Ready to retrain models!")

Saved rich dataset ✅

New train shape: (36497, 10)
New test shape: (9125, 10)

 Ready to retrain models!


In [18]:
# Detect data leakage - find columns that perfectly predict the label
import pandas as pd

df = pd.read_csv(r"C:\Research\EndpointXAI\data\mordor_rich_features.csv")

print("="*60)
print("LEAKAGE DETECTOR")
print("="*60)
print("\nIf any column has values that map 1-to-1 with attack_label,")
print("that's data leakage.\n")

for col in df.columns:
    if col == 'attack_label':
        continue
    
    # For each unique value in this column, check if it always maps to same label
    grouped = df.groupby(col)['attack_label'].nunique()
    
    # If every unique value maps to only 1 label = LEAKAGE
    leakage_score = (grouped == 1).sum() / len(grouped) * 100
    
    print(f"{col}: {leakage_score:.1f}% leakage risk")

LEAKAGE DETECTOR

If any column has values that map 1-to-1 with attack_label,
that's data leakage.

EventID: 67.7% leakage risk
Task: 56.2% leakage risk
Channel: 62.5% leakage risk
SourceName: 62.5% leakage risk
SourceModuleName: 50.0% leakage risk
Severity: 60.0% leakage risk
SeverityValue: 60.0% leakage risk
AccountName: 50.0% leakage risk
AccountType: 33.3% leakage risk
Keywords: 50.0% leakage risk
